# SER Pipeline Checkpoint Analysis

Notebook này dùng để load checkpoint đã train và phân tích từng output của pipeline:

1. Dataset sample: waveform, transcript, label.
2. Feature extractor: `input_values`, `attention_mask`.
3. SSL encoder: `last_hidden_state`.
4. Pooling: mean hoặc attention pooling.
5. Classifier: logits, probabilities, predicted emotion.
6. Acoustic cues và explanation.
7. Evaluation nhanh trên một số sample.

In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import yaml
from IPython.display import Audio, display
from torch.utils.data import DataLoader
from transformers import AutoFeatureExtractor

from dataset import CANONICAL_LABELS, SERDataCollator, load_iemocap_splits
from evaluate import compute_metrics, predict_batches
from features import describe_acoustic_cues, extract_acoustic_features
from inference import make_explanation
from model import SERModel
from train import resolve_device

## 1. Cấu hình đường dẫn

Sửa `CHECKPOINT_PATH` nếu checkpoint nằm ở nơi khác.

In [ ]:
CONFIG_PATH = Path("config.yaml")
CHECKPOINT_PATH = Path("outputs/ser_baseline/best.pt")
DEVICE_NAME = "auto"

device = resolve_device(DEVICE_NAME)
print("device:", device)
print("checkpoint exists:", CHECKPOINT_PATH.exists())

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
config = checkpoint.get("config")
if config is None:
    with open(CONFIG_PATH, "r", encoding="utf-8") as handle:
        config = yaml.safe_load(handle)

model_cfg = config["model"]
audio_cfg = config["audio"]
training_cfg = config["training"]
sampling_rate = int(audio_cfg.get("sampling_rate", 16000))

print("encoder:", model_cfg["encoder_name"])
print("pooling:", model_cfg.get("pooling", "mean"))
print("labels:", CANONICAL_LABELS)
print("checkpoint metrics:")
checkpoint.get("metrics", {})

## 2. Load dataset, feature extractor, model

In [ ]:
datasets = load_iemocap_splits(config)
for split, ds in datasets.items():
    counts = Counter(ds["emotion"])
    print(split, "n=", len(ds), "label_counts=", dict(counts))

In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(model_cfg["encoder_name"])
collator = SERDataCollator(feature_extractor, sampling_rate=sampling_rate)

model = SERModel(
    encoder_name=model_cfg["encoder_name"],
    num_labels=len(CANONICAL_LABELS),
    pooling=model_cfg.get("pooling", "mean"),
    freeze_encoder=False,
    dropout=float(model_cfg.get("dropout", 0.2)),
    hidden_dim=int(model_cfg.get("hidden_dim", 256)),
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(model)

## 3. Chọn một sample để phân tích

Đổi `SPLIT` và `INDEX` để xem utterance khác.

In [ ]:
SPLIT = "validation"
INDEX = 0

row = datasets[SPLIT][INDEX]
waveform = np.asarray(row["input_values"], dtype=np.float32)
transcript = row.get("transcript", "")
true_label_id = int(row["labels"])
true_emotion = CANONICAL_LABELS[true_label_id]

print("split:", SPLIT)
print("index:", INDEX)
print("waveform shape:", waveform.shape)
print("duration seconds:", len(waveform) / sampling_rate)
print("transcript:", transcript)
print("true emotion:", true_emotion)
display(Audio(waveform, rate=sampling_rate))

## 4. Dataset acoustic cues

In [ ]:
dataset_cues = {
    key: row.get(key)
    for key in ["speaking_rate", "pitch_mean", "pitch_std", "rms", "relative_db"]
    if key in row
}
computed_cues = extract_acoustic_features(
    waveform,
    sampling_rate=sampling_rate,
    transcript=transcript,
    dataset_row=row,
)

print("dataset cues:")
print(dataset_cues)
print("computed/merged cues:")
print(computed_cues)
print(describe_acoustic_cues(computed_cues))

## 5. Feature extractor output

In [ ]:
encoded = feature_extractor(
    [waveform],
    sampling_rate=sampling_rate,
    padding=True,
    return_attention_mask=True,
    return_tensors="pt",
)

print("feature extractor keys:", encoded.keys())
for key, value in encoded.items():
    print(key, tuple(value.shape), value.dtype)

print("first 20 input_values:")
encoded["input_values"][0, :20]

## 6. Encoder hidden states

In [ ]:
with torch.no_grad():
    input_values = encoded["input_values"].to(device)
    attention_mask = encoded.get("attention_mask")
    attention_mask = attention_mask.to(device) if attention_mask is not None else None
    encoder_outputs = model.encoder(input_values=input_values, attention_mask=attention_mask)
    hidden_states = encoder_outputs.last_hidden_state

print("last_hidden_state shape:", tuple(hidden_states.shape))
print("hidden mean/std:", float(hidden_states.mean().cpu()), float(hidden_states.std().cpu()))
hidden_states[0, :3, :8].detach().cpu()

## 7. Pooling output

In [ ]:
with torch.no_grad():
    feature_mask = model._feature_attention_mask(attention_mask, hidden_states.shape[1])
    if model.pooling == "attention":
        pooled = model.attention_pooling(hidden_states, feature_mask)
    else:
        pooled = model._mean_pool(hidden_states, feature_mask)

print("feature mask shape:", None if feature_mask is None else tuple(feature_mask.shape))
print("pooled shape:", tuple(pooled.shape))
print("pooled mean/std:", float(pooled.mean().cpu()), float(pooled.std().cpu()))
pooled[0, :16].detach().cpu()

## 8. Classifier logits, probabilities, prediction

In [ ]:
with torch.no_grad():
    logits = model.classifier(pooled)
    probabilities = torch.softmax(logits, dim=-1).squeeze(0).detach().cpu().numpy()
    predicted_id = int(np.argmax(probabilities))
    predicted_emotion = CANONICAL_LABELS[predicted_id]
    confidence = float(probabilities[predicted_id])

print("logits:", logits.squeeze(0).detach().cpu().numpy())
print("probabilities:")
for label, prob in zip(CANONICAL_LABELS, probabilities):
    print(f"  {label:>7}: {prob:.6f}")
print("predicted:", predicted_emotion)
print("confidence:", confidence)
print("true:", true_emotion)

## 9. Explanation output

In [ ]:
explanation = make_explanation(
    predicted_emotion=predicted_emotion,
    confidence=confidence,
    transcript=transcript,
    acoustic_features=computed_cues,
)
print(explanation)

## 10. Một hàm phân tích nhanh sample bất kỳ

In [ ]:
@torch.no_grad()
def analyze_sample(split="validation", index=0, play_audio=False):
    row = datasets[split][index]
    waveform = np.asarray(row["input_values"], dtype=np.float32)
    transcript = row.get("transcript", "")
    true_label = CANONICAL_LABELS[int(row["labels"])]
    encoded = feature_extractor(
        [waveform],
        sampling_rate=sampling_rate,
        padding=True,
        return_attention_mask=True,
        return_tensors="pt",
    )
    logits = model(
        input_values=encoded["input_values"].to(device),
        attention_mask=encoded.get("attention_mask").to(device),
    )
    probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
    pred_id = int(np.argmax(probs))
    cues = extract_acoustic_features(waveform, sampling_rate, transcript=transcript, dataset_row=row)
    result = {
        "split": split,
        "index": index,
        "duration": len(waveform) / sampling_rate,
        "transcript": transcript,
        "true_emotion": true_label,
        "predicted_emotion": CANONICAL_LABELS[pred_id],
        "confidence": float(probs[pred_id]),
        "probabilities": {label: float(probs[i]) for i, label in enumerate(CANONICAL_LABELS)},
        "acoustic_features": cues,
    }
    result["explanation"] = make_explanation(
        result["predicted_emotion"], result["confidence"], transcript, cues
    )
    if play_audio:
        display(Audio(waveform, rate=sampling_rate))
    return result

analyze_sample("validation", 0, play_audio=False)

## 11. Evaluation nhanh trên một subset

Đổi `MAX_EVAL_SAMPLES` nếu muốn đánh giá nhiều hơn.

In [ ]:
EVAL_SPLIT = "validation"
MAX_EVAL_SAMPLES = 64

eval_dataset = datasets[EVAL_SPLIT].select(range(min(MAX_EVAL_SAMPLES, len(datasets[EVAL_SPLIT]))))
eval_loader = DataLoader(
    eval_dataset,
    batch_size=int(training_cfg.get("eval_batch_size", 8)),
    shuffle=False,
    collate_fn=collator,
    num_workers=0,
)

predictions, targets, eval_loss = predict_batches(
    model,
    eval_loader,
    device,
    progress_bar=True,
    description=f"Analyze {EVAL_SPLIT}",
)
metrics = compute_metrics(predictions, targets)
metrics["loss"] = eval_loss
metrics["labels"] = CANONICAL_LABELS
metrics

## 12. Xem các case sai

In [ ]:
wrong = []
for local_idx, (pred, target) in enumerate(zip(predictions.tolist(), targets.tolist())):
    if pred != target:
        row = eval_dataset[local_idx]
        wrong.append(
            {
                "local_idx": local_idx,
                "true": CANONICAL_LABELS[target],
                "pred": CANONICAL_LABELS[pred],
                "transcript": row.get("transcript", ""),
            }
        )

print("num wrong:", len(wrong))
wrong[:10]